# Sample Training Data for Quality Labeling

Samples ~10K documents per 25-year period from **both English corpus and additional news data**,
proportional to their document counts. **Only samples from clean documents** (uses masks from `Clean_Data.ipynb`).

**Prerequisite:** Run `Clean_Data.ipynb` first to generate cleaning masks.

**English:** `D:\hist_LLM\corpus\raw\{year}\subset_*.parquet` (col: `text`, `identifier`)  
**Additional:**
- NYT: `nyt_{year}.parquet` (col: `combined_text`, `_id`)
- Economist: `economist_{year}-*.parquet` (col: `ocr_text`, `article_id`)
- FT: `{year}.parquet` (col: `text_cleaned`, `id`)
- Newswire: `{year}_data_clean.json` (col: `cleaned_article`)

**Output:** `D:\hist_LLM\processing\sample_data\training_samples_{period}.parquet`  
Schema: `[text, original_index, year, source]`

In [ ]:
import pandas as pd
import json
import pyarrow.parquet as pq
from pathlib import Path
import numpy as np
import gc
from tqdm.auto import tqdm

# --- CONFIG ---
ENGLISH_RAW_DIR = Path(r"D:\hist_LLM\corpus\raw")
ENGLISH_MASKS_DIR = Path(r"D:\hist_LLM\corpus\cleaning_masks")
ADDITIONAL_RAW_DIR = Path(r"D:\hist_LLM\additional_data\raw")
ADDITIONAL_MASKS_DIR = Path(r"D:\hist_LLM\additional_data\cleaning_masks")
SAMPLE_OUTPUT_DIR = Path(r"D:\hist_LLM\processing\sample_data")
SAMPLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_SIZE = 10_000
RANDOM_STATE = 42

ADDITIONAL_COLLECTIONS = {
    "nyt": {
        "dir": ADDITIONAL_RAW_DIR / "news_archives" / "NYT_filtered_500char",
        "file_pattern": "nyt_{year}.parquet",
        "text_col": "combined_text",
        "id_col": "_id",
    },
    "economist": {
        "dir": ADDITIONAL_RAW_DIR / "news_archives" / "Economist",
        "file_pattern": "economist_{year}-*.parquet",
        "text_col": "ocr_text",
        "id_col": "article_id",
    },
    "ft": {
        "dir": ADDITIONAL_RAW_DIR / "news_archives" / "FT",
        "file_pattern": "{year}.parquet",
        "text_col": "text_cleaned",
        "id_col": "id",
    },
    "newswire": {
        "dir": ADDITIONAL_RAW_DIR / "newswire",
        "file_pattern": "{year}_data_clean.json",
        "text_col": "cleaned_article",
        "id_col": None,
    },
}


# ======================== COUNTING ========================

def count_english_clean(year: int) -> int:
    """Count clean docs for a year using cleaning masks."""
    mask_dir = ENGLISH_MASKS_DIR / str(year)
    if not mask_dir.exists():
        return 0
    total = 0
    for f in mask_dir.glob("*_mask.parquet"):
        try:
            total += pq.ParquetFile(f).metadata.num_rows
        except:
            pass
    return total


def count_additional_clean(year: int) -> dict:
    """Count clean docs per additional collection for a year."""
    counts = {}
    for name, cfg in ADDITIONAL_COLLECTIONS.items():
        coll_mask_dir = ADDITIONAL_MASKS_DIR / name
        if name == "newswire":
            mask_path = coll_mask_dir / f"{year}_mask.parquet"
            if mask_path.exists():
                counts[name] = pq.ParquetFile(mask_path).metadata.num_rows
        elif name == "economist":
            total = 0
            for f in coll_mask_dir.glob(f"economist_{year}-*_mask.parquet"):
                total += pq.ParquetFile(f).metadata.num_rows
            if total > 0:
                counts[name] = total
        elif name == "nyt":
            mask_path = coll_mask_dir / f"nyt_{year}_mask.parquet"
            if mask_path.exists():
                counts[name] = pq.ParquetFile(mask_path).metadata.num_rows
        elif name == "ft":
            mask_path = coll_mask_dir / f"{year}_mask.parquet"
            if mask_path.exists():
                counts[name] = pq.ParquetFile(mask_path).metadata.num_rows
    return counts


# ======================== SAMPLING ========================

def sample_english(year: int, n: int, rng: np.random.Generator) -> list:
    """Sample n clean docs from English raw data for a year."""
    year_dir = ENGLISH_RAW_DIR / str(year)
    mask_dir = ENGLISH_MASKS_DIR / str(year)
    if not year_dir.exists() or not mask_dir.exists():
        return []

    files = sorted(year_dir.glob("subset_*.parquet"))
    if not files:
        return []

    # Build global list of (file, clean_row_indices) using masks
    file_clean_indices = []
    for f in files:
        mask_path = mask_dir / f"{f.stem}_mask.parquet"
        if mask_path.exists():
            mask_df = pd.read_parquet(mask_path)
            clean_idx = set(mask_df["original_index"].tolist())
            if clean_idx:
                file_clean_indices.append((f, clean_idx))

    total_clean = sum(len(idx) for _, idx in file_clean_indices)
    if total_clean == 0:
        return []

    n = min(n, total_clean)

    # Generate random global indices, map to per-file
    chosen = set(rng.choice(total_clean, size=n, replace=False))
    samples = []
    offset = 0
    for f, clean_idx in file_clean_indices:
        sorted_idx = sorted(clean_idx)
        file_chosen = [sorted_idx[i - offset] for i in chosen if offset <= i < offset + len(sorted_idx)]
        offset += len(sorted_idx)
        if not file_chosen:
            continue
        df = pd.read_parquet(f, columns=["identifier", "text"])
        selected = df.loc[df.index.isin(file_chosen)]
        for _, row in selected.iterrows():
            text = row["text"]
            if text and str(text).strip():
                samples.append({
                    "text": text,
                    "original_index": row["identifier"],
                    "year": year,
                    "source": "english",
                })
        del df, selected
    gc.collect()
    return samples


def sample_additional(year: int, collection: str, n: int, rng: np.random.Generator) -> list:
    """Sample n clean docs from an additional data collection for a year."""
    cfg = ADDITIONAL_COLLECTIONS[collection]
    coll_mask_dir = ADDITIONAL_MASKS_DIR / collection

    # Load clean indices from mask
    if collection == "newswire":
        mask_path = coll_mask_dir / f"{year}_mask.parquet"
        if not mask_path.exists() or year == 1957:
            return []
        clean_idx = set(pd.read_parquet(mask_path)["original_index"].tolist())
        path = cfg["dir"] / f"{year}_data_clean.json"
        if not path.exists():
            return []
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        valid = [(i, data[i].get("cleaned_article", "")) for i in clean_idx if i < len(data)]
        valid = [(i, t) for i, t in valid if t]
        n = min(n, len(valid))
        chosen = rng.choice(len(valid), size=n, replace=False)
        return [{
            "text": valid[c][1],
            "original_index": f"newswire_{year}_{valid[c][0]}",
            "year": year,
            "source": "newswire",
        } for c in chosen]

    elif collection == "economist":
        files = sorted(cfg["dir"].glob(f"economist_{year}-*.parquet"))
        if not files:
            return []
        # Load all masks for this year's weekly files
        all_dfs = []
        for f in files:
            mask_path = coll_mask_dir / f"{f.stem}_mask.parquet"
            if mask_path.exists():
                clean_idx = set(pd.read_parquet(mask_path)["original_index"].tolist())
                df = pd.read_parquet(f, columns=[cfg["id_col"], cfg["text_col"]])
                df = df.loc[df.index.isin(clean_idx)]
                df["_src_file"] = f.name
                all_dfs.append(df)
        if not all_dfs:
            return []
        df = pd.concat(all_dfs, ignore_index=True)
    else:
        pattern = cfg["file_pattern"].format(year=year)
        path = cfg["dir"] / pattern
        if not path.exists():
            return []
        mask_path = coll_mask_dir / f"{Path(pattern).stem}_mask.parquet"
        if not mask_path.exists():
            return []
        clean_idx = set(pd.read_parquet(mask_path)["original_index"].tolist())
        df = pd.read_parquet(path, columns=[cfg["id_col"], cfg["text_col"]])
        df = df.loc[df.index.isin(clean_idx)]

    df = df.dropna(subset=[cfg["text_col"]])
    df = df[df[cfg["text_col"]].str.strip() != ""]
    if df.empty:
        return []
    n = min(n, len(df))
    sampled = df.sample(n=n, random_state=rng.integers(2**31))
    return [{
        "text": row[cfg["text_col"]],
        "original_index": str(row[cfg["id_col"]]),
        "year": year,
        "source": collection,
    } for _, row in sampled.iterrows()]


# ======================== MAIN ========================

def collect_combined_samples(target_years: list):
    """Sample 10K clean docs from English + additional data, proportional to doc counts."""
    rng = np.random.default_rng(RANDOM_STATE)

    # Phase 1: Count clean docs per source per year
    print("Counting clean documents per source...", flush=True)
    year_english = {}
    year_additional = {}

    for year in tqdm(target_years, desc="Counting"):
        year_english[year] = count_english_clean(year)
        year_additional[year] = count_additional_clean(year)

    total_english = sum(year_english.values())
    total_additional = sum(sum(c.values()) for c in year_additional.values())
    total_all = total_english + total_additional

    if total_all == 0:
        print("No clean documents found! Run Clean_Data.ipynb first.")
        return

    n_english = round(SAMPLE_SIZE * total_english / total_all)
    n_additional = SAMPLE_SIZE - n_english

    print(f"\nClean docs: {total_all:,} (English: {total_english:,}, Additional: {total_additional:,})")
    print(f"Sample split: {n_english} English + {n_additional} additional = {SAMPLE_SIZE}")

    # Phase 2: Sample English
    print("\nSampling English...", flush=True)
    all_samples = []
    for year in tqdm(target_years, desc="English"):
        if year_english[year] == 0:
            continue
        year_n = max(1, round(n_english * year_english[year] / total_english))
        samples = sample_english(year, year_n, rng)
        all_samples.extend(samples)

    # Phase 3: Sample additional
    if n_additional > 0:
        print("Sampling additional data...", flush=True)
        add_entries = []
        for year in target_years:
            for coll, count in year_additional[year].items():
                add_entries.append((year, coll, count))
        total_add_counted = sum(c for _, _, c in add_entries)
        for year, coll, count in tqdm(add_entries, desc="Additional"):
            year_coll_n = max(1, round(n_additional * count / total_add_counted))
            samples = sample_additional(year, coll, year_coll_n, rng)
            all_samples.extend(samples)

    # Phase 4: Shuffle and save
    start_year = min(target_years)
    end_year = max(target_years)
    output_path = SAMPLE_OUTPUT_DIR / f"training_samples_{start_year}_{end_year}.parquet"

    final_df = pd.DataFrame(all_samples)
    final_df = final_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    final_df.to_parquet(output_path, index=False)

    source_counts = final_df["source"].value_counts()
    print(f"\nSaved {len(final_df)} samples to {output_path.name}")
    print("Source breakdown:")
    for src, cnt in source_counts.items():
        print(f"  {src}: {cnt} ({100*cnt/len(final_df):.1f}%)")

In [ ]:
# Generate all 14 periods and sample each
all_periods = [range(1678, 1701)]
for start in range(1701, 2002, 25):
    end = min(start + 24, 2023)
    all_periods.append(range(start, end + 1))

# Run for all periods (or subset as needed)
for period_range in all_periods:
    print(f"\n{'='*60}")
    print(f"Processing Period: {min(period_range)}_{max(period_range)}")
    print(f"{'='*60}")
    collect_combined_samples(list(period_range))